# run_train_cnn: ResNet-18 on square crops, three experiments (M3)

Colab-ready (Runtime -> Change runtime type -> GPU). Not executed in CI.
Experiments (README plan): same val/test sessions, different train sessions:

| run | train sessions |
|---|---|
| `sim_only` | Gazebo sessions only |
| `real_only` | real capture sessions only |
| `mixed` | sim + real |

**TODO**
- [ ] training / validation steps in `train.ipynb: train_cnn`
- [ ] upload `data/crops/` to Colab (Drive mount or zip) and fill the session lists below

## 1. Setup (Colab only; skip locally)

In [ ]:
import os
IN_COLAB = "COLAB_GPU" in os.environ or os.path.exists("/content")
if IN_COLAB:
    !git clone https://github.com/BrandonZabawa/Gambit-the-robot-chess-player.git
    %cd Gambit-the-robot-chess-player/gambit_vision/notebooks
    !pip install -q torch torchvision tensorboard chess pyyaml pandas scikit-learn opencv-contrib-python
    # TODO: get the crops, e.g.
    # from google.colab import drive; drive.mount("/content/drive")
    # !mkdir -p ../data && cp -r /content/drive/MyDrive/gambit/crops ../data/crops

In [ ]:
import torch
print("torch", torch.__version__, "cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
%run ./train.ipynb
%cd ..

## 2. Experiments

In [ ]:
SIM = []       # TODO: e.g. ["sim_01", "sim_02"]
REAL = []      # TODO: e.g. ["s01_daylight", "s02_lamp"]
VAL = []       # TODO: real sessions, disjoint from REAL
TEST = []      # TODO: real, held out from everything

EXPERIMENTS = {"sim_only": SIM, "real_only": REAL, "mixed": SIM + REAL}
checkpoints = {}
for run_name, train_sessions in EXPERIMENTS.items():
    cfg = load_config("configs/train.yaml")
    cfg.data.train_sessions, cfg.data.val_sessions, cfg.data.test_sessions = train_sessions, VAL, TEST
    checkpoints[run_name] = train_cnn(cfg, run_name=run_name)
    print(run_name, "->", checkpoints[run_name])

## 3. TensorBoard (optional)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir models/resnet18